In [ ]:
%pip install -e ../../../

In [ ]:
import pandas as pd

pump_options_df = pd.read_excel('../pump_options-static_properties.xlsx', index_col='option_id')
COLUMNS = list(pump_options_df.index)
Y2000 = pd.Timestamp(year=2000, month=1, day=1)
Y2024 = pd.Timestamp(year=2024, month=1, day=1)

from water_futures_battle.economy.dynamic_properties import EconomyDB
from water_futures_battle.economy.services import adjust_for_inflation

econ_db = EconomyDB.load_from_file('../../economy/economy-dynamic_properties.xlsx')
inflation = econ_db[EconomyDB.INFLATION]['NL0000']
inflation = inflation[inflation.index <= Y2024]

COSTS_2024 = [
    5000,
    7000,
    8000,
    10000
]

costs_df = pd.DataFrame(
    data=[COSTS_2024],
    columns=COLUMNS,
    index=pd.DatetimeIndex([Y2024], name='timestamp')
)

costs_df = adjust_for_inflation(costs_df.loc[Y2024], Y2024, inflation)

In [ ]:
from water_futures_battle.core.base_model import DynamicProperties

pump_options_dps = DynamicProperties('pump_options-dynamic_properties')
pump_options_dps['new_pump-cost'] = costs_df.loc[[Y2000]].copy()

pump_options_dps.dump(path='..')